# Interactive Sheep Data Explorer

This notebook allows you to explore averaged frames from the sheep PPG dataset interactively. 

Select an experiment index (0-12), choose a wavelength (WV1 or WV2) and channel (1-5), and the notebook will:
1. Load the corresponding sheep dataframe
2. Extract the selected channel data
3. Compute averaged frames using the heart rate information
4. Display the first 12 averaged frames in a 4x3 subplot grid

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import ipywidgets as widgets
from IPython.display import display, clear_output
from psafe_custom import (
    get_sheep_dataframe,
    get_sorted_ac_columns,
    compute_frames_from_df
)

## Define SQI
(Skenewss - "It is observed that the skew and kurtosis measured for the case of motion-corrupted data are much higher in magnitude when compared to the skew and kurtosis for clean data.", in 'Two-Stage Approach for Detection and Reduction of Motion Artifacts in Photoplethysmographic Data' - by Rajet Krishnan )

In [3]:
from scipy.stats import skew

def sqi(sig_1D: np.ndarray) -> float:
    # mean = np.mean(sig_1D)
    # std_dev = np.std(sig_1D)
    # return float(np.sum((sig_1D - mean)/std_dev)) **3 / len(sig_1D)
    return skew(sig_1D)


## Define Interactive Dropdown Widgets

Create dropdowns to select the experiment index and the wavelength/channel combination.

In [ ]:
# Create dropdown for experiment index (0-12)
index_dropdown = widgets.Dropdown(
    options=[(f"Experiment {i}", i) for i in range(13)],
    value=0,
    description="Experiment Index:",
    disabled=False,
)

# Create dropdown for wavelength/channel combination
# Generate all combinations: WV1-Ch1, WV1-Ch2, ..., WV2-Ch5
wv_channel_options = []
for wv in ['wv1', 'wv2']:
    for ch in range(1, 6):
        label = f"{wv.upper()}-Channel{ch}"
        value = (wv, ch)
        wv_channel_options.append((label, value))

wv_channel_dropdown = widgets.Dropdown(
    options=wv_channel_options,
    value=('wv1', 1),
    description="Wavelength/Channel:",
    disabled=False,
)

## Process and Visualize Data

The cell below loads the data, computes averaged frames, and displays them in a 4x3 subplot grid.

In [ ]:
# Global cache to store computed frames for each sheep index
frames_cache = {}

def compute_and_cache_all_frames(sheep_idx):
    """
    Load sheep data and compute averaged frames for ALL wavelength/channel combinations.
    Store results in frames_cache keyed by sheep index.
    """
    if sheep_idx in frames_cache:
        print(f"Using cached frames for experiment {sheep_idx}")
        return frames_cache[sheep_idx]
    
    print(f"Loading and computing frames for experiment {sheep_idx}...")
    
    # Load the sheep dataframe
    data = get_sheep_dataframe(sheep_idx)
    print(f"Loaded data with shape: {data.shape}")
    
    # Get all AC column names
    ac_columns = get_sorted_ac_columns()
    
    # Compute frames for all columns at once (more efficient)
    all_columns = ac_columns['wv1'] + ac_columns['wv2']
    
    print(f"Computing frames for {len(all_columns)} channels...")
    frames_dict = compute_frames_from_df(
        dataframe=data,
        time_series_columns=all_columns,
        heartrate_column='FHR',
        sampling_rate=80.0,
        average_frame_length=100,
        frames_to_average=120,
        sample_skip=80,
        interpolation_kind="linear"
    )
    
    # Cache the results
    frames_cache[sheep_idx] = frames_dict
    print(f"Cached frames for {len(frames_dict)} channels")
    
    return frames_dict


def plot_frames(sheep_idx, wv_channel):
    """
    Plot the selected frames from cache. Ensures frames are computed if needed.
    
    Parameters:
    - sheep_idx: experiment index (0-12)
    - wv_channel: tuple of (wv, channel_number) e.g., ('wv1', 1)
    """
    wv, channel_num = wv_channel
    
    # Compute/retrieve frames for this sheep index
    frames_dict = compute_and_cache_all_frames(sheep_idx)
    
    # Get AC column names to find the right key
    ac_columns = get_sorted_ac_columns()
    selected_column = ac_columns[wv][channel_num - 1]  # channel is 1-indexed
    print(f"Plotting {selected_column}")
    
    # Retrieve frames from cache
    frames = frames_dict[selected_column]
    print(f"Retrieved {len(frames)} averaged frames")
    
    # Create 4x3 subplot grid for first 12 frames
    fig, axes = plt.subplots(4, 3, figsize=(14, 10), sharex=True, sharey=False)
    axes = axes.flatten()  # Flatten to easily iterate
    
    # Get the first 12 frames or fewer if not available
    num_frames_to_plot = min(12, len(frames))
    
    for i in range(num_frames_to_plot):
        axes[i].plot(frames[i], linewidth=1.5, color='steelblue')
        frame_sqi = sqi(frames[i])
        axes[i].set_title(f'Frame {i + 1}, SQI: {frame_sqi}', fontsize=10)
        axes[i].grid(True, alpha=0.3)
    
    # Hide unused subplots
    for i in range(num_frames_to_plot, 12):
        axes[i].axis('off')
    
    # Add labels
    fig.text(0.5, 0.04, 'Sample', ha='center', fontsize=12)
    fig.text(0.04, 0.5, 'Amplitude', va='center', rotation='vertical', fontsize=12)
    fig.suptitle(f'First {num_frames_to_plot} Averaged Frames - {wv.upper()}, Channel {channel_num}', 
                 fontsize=14, fontweight='bold')
    
    plt.tight_layout(rect=[0.05, 0.05, 1, 0.96])
    plt.show()


# Create interactive plot using interact
from ipywidgets import interact

interact(
    plot_frames,
    sheep_idx=index_dropdown,
    wv_channel=wv_channel_dropdown
)

interactive(children=(Dropdown(description='Experiment Index:', options=(('Experiment 0', 0), ('Experiment 1',…

<function __main__.plot_frames(sheep_idx, wv_channel)>

## Instructions

1. **Select Experiment Index**: Use the first dropdown to choose an experiment from 0 to 12
   - When you change this, all frames for all 10 wavelength-channel combinations are computed and cached
2. **Select Wavelength and Channel**: Use the second dropdown to select from WV1 (wavelength 1) or WV2 (wavelength 2), each with channels 1-5
   - When you change this, the plot updates instantly from the cached frames (no recomputation!)
3. **View Results**: The plot shows the first 12 averaged frames in a 4x3 grid
4. **Shared Axes**: All subplots share the same x and y axes for easy comparison across frames

### Frame Parameters
- **Average Frame Length**: 100 samples
- **Frames to Average**: 120 consecutive frames averaged together
- **Sample Skip**: 80 samples between consecutive averaged frames
- **Sampling Rate**: 80 Hz (sheep PPG data)
- **Interpolation**: Linear interpolation for variable frame lengths

### Performance Note
- Frames are computed once per sheep index and cached in memory
- Changing wavelength/channel is instant since it just retrieves cached data
- Changing sheep index triggers a full recomputation of all 10 wavelength-channel combinations